In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import scrublet as scr
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rcParams
from matplotlib import rc
from matplotlib import colors
import seaborn as sb
from collections import Counter
import glob

sc.settings.set_figure_params(dpi = 160, color_map = 'RdPu', dpi_save = 300, format = 'pdf')
plt.rcParams["figure.figsize"] = [6,6]
plt.rcParams["pdf.fonttype"] = 42 # add pdf text as one piece instead of letter by letter

In [2]:
import importlib.metadata as md
print(md.version("scrublet"))

0.2.3


In [3]:
sb.set_palette('colorblind')

In [4]:
# set working directory
%cd /nfs/team205/cs42/HSC/HSC_new_to_transfer/

/nfs/team205/cs42/HSC/HSC_new_to_transfer


In [5]:
# figure directory
fig_path = 'figure/'
sc.settings.figdir = fig_path

# Download dataset

In [6]:
manifest = pd.read_csv('manifest/sample_manifest_01052026_manuscript.csv')
manifest = manifest.loc[manifest['screen_no'].isin(['SPI1_inducible']),:]

In [7]:
smp_dict = list(manifest['sequence_name'])

In [8]:
smp_dict

['6464STDY15422806', '6464STDY15422807']

In [9]:
# check whether all files have been downloaded in matrix folder - checked - all downloaded from new mapping cellranger 720
DATAFOLDER = '/nfs/team205/cs42/HSC_matrix/'
exist = [x.split('/')[-1] for x in glob.glob(DATAFOLDER+'*')]

if len([x for x in smp_dict if x not in exist])>0:
    print([smp_dict[x][2] for x in smp_dict if x not in exist],'not in matrix folder')

In [12]:
def read_files(filename, raw_file_path = '/nfs/team205/cs42/HSC_matrix/',
               min_n_count = 1000, min_n_gene = 500):

    
    path = '%s/%s/'%(raw_file_path,filename)
    adata = sc.read_10x_h5(path+'filtered_feature_bc_matrix.h5', gex_only=False)
    
    adata.obs_names = [filename+"-"+x.strip("-1") for x in adata.obs_names]
   
    # define mito & GEX
    gex = np.array(adata.var['feature_types']=='Gene Expression')
    mito_genes = np.array(adata.var_names.str.startswith('MT-')) & gex
    exog_genes = np.array(adata.var_names.str.startswith('EXOG_')) 
    # update gex to exclude exog_genes
    gex_update = gex & ~exog_genes
    
    antibody_capture = np.array(adata.var['feature_types']=='Antibody Capture')
    
    # caculate n_counts / n_genes per cell
    adata.obs['n_counts'] = np.sum(adata[:,gex_update].X, axis=1).A1
    adata.obs['n_genes'] = np.sum(adata[:,gex_update].X>0,axis=1)
    adata.obs['n_counts_antibody_capture'] = np.sum(adata[:,antibody_capture].X, axis=1).A1
    
    # calculate mito%
    adata.obs['mito'] = (np.sum(adata.X[:, mito_genes],axis=1).A1) / (np.sum(adata[:,gex_update].X,axis=1).A1)
    
    
    # filter cells
    clist = []
    clist.append(np.array(adata.obs['n_counts'] > 1000))
    clist.append(np.array(adata.obs['n_genes'] > 500))
    #clist.append(np.array(adata.obs['n_genes'] < 7000))
    clist.append(np.array(adata.obs['mito'] < 0.2)) # filter out mito >= 0.2
    
    c = np.column_stack(clist).all(axis=1)
    adata = adata[c].copy()
    
    # update manifest onto .obs
    adata.obs['file'] = filename
    adata.obs['sanger_sample_ID'] = manifest.loc[manifest['sequence_name']==filename, 'sanger_sample_ID'].item()
    adata.obs['screen_no'] = manifest.loc[manifest['sequence_name']==filename, 'screen_no'].item()
    adata.obs['Date_of_sample'] = manifest.loc[manifest['sequence_name']==filename, 'Date_of_sample'].item()
    adata.obs['HSC_days'] = manifest.loc[manifest['sequence_name']==filename, 'HSC days'].item()
    adata.obs['FACS_sort'] = manifest.loc[manifest['sequence_name']==filename, 'FACS_sort'].item()
    adata.obs['10x_kit'] = manifest.loc[manifest['sequence_name']==filename, '10x kit'].item()
    
    # calculate doublets using scrublet
    scrub = scr.Scrublet(adata.X)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=False)
    adata.obs['doublet_scores'] = doublet_scores
    adata.obs['predicted_doublets'] = predicted_doublets
    
    sc.write('%s_filtered'%(filename),adata)

In [13]:
from multiprocessing import Pool

with Pool(10) as p:
    p.map(read_files, [x for x in smp_dict])

/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


# Merge dataset

In [14]:
ad = {}

In [15]:
for filename in smp_dict:
    ad[filename] = sc.read('%s_filtered'%(filename))
    ad[filename].var_names_make_unique()

/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/anndata/_core/anndata.py:1776: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [16]:
merged = None;
for key in ad: 
    if merged is None:
        merged = ad[key]
    else:
        merged = merged.concatenate(ad[key], join='outer')

/tmp/ipykernel_2871821/1590458944.py:6: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  merged = merged.concatenate(ad[key], join='outer')


In [17]:
Counter(merged.obs['file'])

Counter({'6464STDY15422806': 2139, '6464STDY15422807': 314})

### Remove doublets

In [18]:
merged = merged[merged.obs['doublet_scores']<0.4]
merged = merged[merged.obs['predicted_doublets']==False]

In [19]:
Counter(merged.obs['file'])

Counter({'6464STDY15422806': 2105, '6464STDY15422807': 312})

In [20]:
merged.write_h5ad('write/SPI1_INDUCIBLE_data_raw_count.h5ad')

## Preprocessing - normalise and log

In [29]:
merged = sc.read('write/SPI1_INDUCIBLE_data_raw_count.h5ad')

In [21]:
sc.pp.normalize_per_cell(merged,counts_per_cell_after=1e4,key_n_counts='n_counts')

/tmp/ipykernel_2871821/4036594748.py:1: FutureWarning: Use sc.pp.normalize_total instead
  sc.pp.normalize_per_cell(merged,counts_per_cell_after=1e4,key_n_counts='n_counts')
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:588: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[key_n_counts] = counts_per_cell
/nfs/users/nfs_c/cs42/.conda/envs/myclone/lib/python3.11/site-packages/scanpy/preprocessing/_simple.py:591: FutureWarning: Use sc.pp.normalize_total instead
  normalize_per_cell(


In [22]:
sc.pp.log1p(merged)

In [23]:
merged.write_h5ad('write/SPI1_INDUCIBLE_data_normalised_log.h5ad')